# 01 · Tratamiento estadístico común (secciones 5.2 a 5.7)

- **Conceptos básicos:** población, muestra, parámetro y estadístico; clasificación de las 6 variables.
- **Datos agrupados de una sola variable: `VAL_CFM` (cargo fijo mensual, COP).** Rango, Sturges, amplitud,
  intervalos, marcas y frecuencias; media, mediana, moda, cuartiles, varianza, desviación estándar y coeficiente de
  variación agrupados, comparados con los valores exactos sobre los datos individuales.
- **ANTIGUEDAD_MESES** como variable discreta (sin agrupar en intervalos continuos).
- **Proporciones** de las banderas digitales y de la restricción de contacto.

Cada medida se calcula dos veces: "a mano" (la fórmula del documento, paso a paso) y con la función de la
biblioteca. La comparación alimenta la tabla de validación de la sección 6.4.

In [1]:
import math
import statistics
import numpy as np
import pandas as pd
import config

el = config.leer("elegibles")
prep = config.leer_json("r00_preparacion")
N_recibidos, n = prep["recibidos"], len(el)
print(f"Registros recibidos: {N_recibidos:,} · muestra elegible analizada: n = {n:,}")
validacion = []   # filas de la tabla 6.4


def validar(indicador, manual, computacional, tolerancia, observacion=""):
    dif = abs(manual - computacional)
    validacion.append({"indicador": indicador, "manual": manual, "computacional": computacional, "diferencia": dif,
                       "tolerancia": tolerancia, "cumple": bool(dif <= tolerancia), "observacion": observacion})

Registros recibidos: 9,000 · muestra elegible analizada: n = 5,230


## 1. Conceptos básicos: población, muestra, parámetro y estadístico (5.2)
- **Población:** registros elegibles de líneas móviles de personas naturales con `FECHA_ACTUALIZACION` entre enero y
  octubre de 2026. Su tamaño **N** en la base completa de la empresa no se conoce: solo se recibió una muestra.
- **Muestra:** los registros elegibles recibidos (**n = 5.230**), seleccionados por mes (1.000 por mes recibidos,
  9 meses, sin septiembre).
- **Parámetro vs. estadístico:** el cargo fijo promedio de toda la población sería μ (desconocido); el de los
  5.230 registros es x̄, un estadístico.

In [2]:
clasificacion = pd.DataFrame([
    ("ANTIGUEDAD_MESES", "Cuantitativa discreta", "Razón", "meses enteros (0 = línea nueva)"),
    ("VAL_CFM", "Cuantitativa monetaria", "Razón", "pesos colombianos; 0 = sin cargo fijo"),
    ("BAN_BANCA", "Cualitativa dicotómica", "Nominal", "True / False"),
    ("BAN_BILLETERA", "Cualitativa dicotómica", "Nominal", "True / False"),
    ("BAN_PAGOS_ONLINE", "Cualitativa dicotómica", "Nominal", "True / False"),
    ("BAN_SUPRESION_I", "Cualitativa dicotómica", "Nominal", "True = restricción de mensajes activa"),
], columns=["variable", "tipo", "nivel", "codificacion_verificada"])
clasificacion["validos"] = [int(el[v].notna().sum()) for v in clasificacion.variable]
clasificacion["distintos"] = [int(el[v].nunique()) for v in clasificacion.variable]
clasificacion

,variable,tipo,nivel,codificacion_verificada,validos,distintos
0,ANTIGUEDAD_MESES,Cuantitativa discreta,Razón,meses enteros (0 = línea nueva),5227,190
1,VAL_CFM,Cuantitativa monetaria,Razón,pesos colombianos; 0 = sin cargo fijo,5230,135
2,BAN_BANCA,Cualitativa dicotómica,Nominal,True / False,5230,2
3,BAN_BILLETERA,Cualitativa dicotómica,Nominal,True / False,5230,2
4,BAN_PAGOS_ONLINE,Cualitativa dicotómica,Nominal,True / False,5230,2
5,BAN_SUPRESION_I,Cualitativa dicotómica,Nominal,True = restricción de mensajes activa,5230,2


## 2. Distribución de datos agrupados de VAL_CFM (5.4)
### Rango, número de clases (Sturges) y amplitud

In [3]:
x = el["VAL_CFM"].dropna().astype(float).to_numpy()
n_cfm = len(x)
xmin, xmax = x.min(), x.max()
R = xmax - xmin
k_sturges = 1 + 3.322 * math.log10(n_cfm)
k = round(k_sturges)
A = math.ceil(R / k)              # amplitud entera en pesos; garantiza que k·A ≥ R
print(f"n = {n_cfm:,} · Xmin = {xmin:,.0f} · Xmax = {xmax:,.0f} · R = {R:,.0f}")
print(f"k = 1 + 3,322·log10({n_cfm}) = {k_sturges:.3f} → k = {k} · A = ⌈R/k⌉ = ⌈{R:,.0f}/{k}⌉ = {A:,}")

n = 5,230 · Xmin = 0 · Xmax = 97,392 · R = 97,392
k = 1 + 3,322·log10(5230) = 13.353 → k = 13 · A = ⌈R/k⌉ = ⌈97,392/13⌉ = 7,492


### Intervalos [Lᵢ, Uᵢ), marcas de clase y frecuencias
El último intervalo incluye el máximo.

In [4]:
L = xmin + A * np.arange(k)
U = L + A
m = (L + U) / 2
clase = np.minimum(((x - xmin) // A).astype(int), k - 1)
f = np.bincount(clase, minlength=k)
tabla = pd.DataFrame({"clase": np.arange(1, k + 1), "L": L, "U": U, "marca": m, "f": f})
tabla["h"] = tabla.f / n_cfm
tabla["F"] = tabla.f.cumsum()
tabla["H"] = tabla.F / n_cfm
assert tabla.f.sum() == n_cfm and abs(tabla.h.sum() - 1) < 1e-12
validar("Σ fᵢ = n", float(tabla.f.sum()), float(n_cfm), 0, "suma de frecuencias absolutas")
validar("Σ hᵢ = 1", float(tabla.h.sum()), 1.0, 1e-9, "suma de frecuencias relativas")
tabla.style.format({"L": "{:,.0f}", "U": "{:,.0f}", "marca": "{:,.1f}", "h": "{:.4f}", "H": "{:.4f}"})

,clase,L,U,marca,f,h,F,H
0,1,0,"7,492","3,746.0",54,0.0103,54,0.0103
1,2,"7,492","14,984","11,238.0",11,0.0021,65,0.0124
2,3,"14,984","22,476","18,730.0",63,0.0120,128,0.0245
3,4,"22,476","29,968","26,222.0",171,0.0327,299,0.0572
4,5,"29,968","37,460","33,714.0",539,0.1031,838,0.1602
5,6,"37,460","44,952","41,206.0",4183,0.7998,5021,0.9600
6,7,"44,952","52,444","48,698.0",166,0.0317,5187,0.9918
7,8,"52,444","59,936","56,190.0",14,0.0027,5201,0.9945
8,9,"59,936","67,428","63,682.0",12,0.0023,5213,0.9967
9,10,"67,428","74,920","71,174.0",13,0.0025,5226,0.9992


### Medidas de tendencia central, posición y dispersión (agrupadas)

In [5]:
F_ant = np.concatenate([[0], tabla.F.to_numpy()[:-1]])   # Fᵢ₋₁


def clase_de(posicion):
    return int(np.searchsorted(tabla.F.to_numpy(), posicion, side="left"))


def cuantil_agrupado(p):
    pos = p * n_cfm
    i = clase_de(pos)
    return {"p": p, "posicion": pos, "clase": i + 1, "L": L[i], "F_ant": int(F_ant[i]), "f": int(f[i]), "A": A,
            "valor": L[i] + (pos - F_ant[i]) / f[i] * A}


# Media agrupada: x̄ ≈ Σ fᵢ·mᵢ / n (a mano, término a término)
suma_fm = 0.0
for fi, mi in zip(f, m):
    suma_fm += fi * mi
media_g = suma_fm / n_cfm
# Mediana y cuartiles por interpolación
Q = {nombre: cuantil_agrupado(p) for nombre, p in [("Q1", .25), ("Q2", .5), ("Q3", .75)]}
mediana_g = Q["Q2"]["valor"]
# Moda agrupada: Mo = L + d1/(d1+d2)·A
i_mod = int(np.argmax(f))
f_prev = f[i_mod - 1] if i_mod > 0 else 0
f_next = f[i_mod + 1] if i_mod < k - 1 else 0
d1, d2 = f[i_mod] - f_prev, f[i_mod] - f_next
moda_unica = (f == f[i_mod]).sum() == 1
moda_g = L[i_mod] + d1 / (d1 + d2) * A if (d1 + d2) > 0 and moda_unica else None
# Varianza, desviación y coeficiente de variación agrupados
suma_cuad = 0.0
for fi, mi in zip(f, m):
    suma_cuad += fi * (mi - media_g) ** 2
var_g = suma_cuad / (n_cfm - 1)
sd_g = math.sqrt(var_g)
cv_g = sd_g / media_g * 100

agrupadas = {"media": media_g, "mediana": mediana_g, "moda": moda_g, "Q1": Q["Q1"]["valor"], "Q2": mediana_g,
             "Q3": Q["Q3"]["valor"], "varianza": var_g, "desviacion": sd_g, "cv": cv_g,
             "suma_fm": suma_fm, "suma_cuad": suma_cuad,
             "moda_detalle": {"clase": i_mod + 1, "L": L[i_mod], "f": int(f[i_mod]), "f_prev": int(f_prev),
                              "f_next": int(f_next), "d1": int(d1), "d2": int(d2), "A": A},
             "cuantiles": Q}
pd.Series({k_: v for k_, v in agrupadas.items() if not isinstance(v, dict)}).round(2)

media         3.966176e+04
mediana       4.064271e+04
moda          4.102361e+04
Q1            3.830090e+04
Q2            4.064271e+04
Q3            4.298452e+04
varianza      4.057385e+07
desviacion    6.369760e+03
cv            1.606000e+01
suma_fm       2.074310e+08
suma_cuad     2.121607e+11
dtype: float64

### Las mismas medidas con funciones de biblioteca (para validar)

In [6]:
media_g_np = np.average(m, weights=f)
var_g_np = np.sum(f * (m - media_g_np) ** 2) / (n_cfm - 1)
validar("Media agrupada", media_g, media_g_np, 1e-6, "lazo manual vs np.average con pesos")
validar("Varianza agrupada", var_g, var_g_np, 1e-3, "lazo manual vs operación vectorizada")
repetidos = np.repeat(m, f)                      # cada registro reemplazado por su marca de clase
validar("Media agrupada vs. marcas repetidas", media_g, float(repetidos.mean()), 1e-6, "np.repeat de las marcas")

### Valores exactos sobre los datos individuales (y diferencia con lo agrupado)
Cuartiles exactos con `numpy.quantile(method="linear")` (el método por defecto de numpy y pandas).

In [7]:
exactas = {
    "media": float(np.mean(x)), "mediana": float(np.median(x)), "moda": float(statistics.mode(x)),
    "Q1": float(np.quantile(x, .25)), "Q2": float(np.quantile(x, .5)), "Q3": float(np.quantile(x, .75)),
    "varianza": float(np.var(x, ddof=1)), "desviacion": float(np.std(x, ddof=1)),
}
exactas["cv"] = exactas["desviacion"] / exactas["media"] * 100
# media y varianza "a mano" sobre los datos individuales
media_manual = sum(x) / n_cfm
var_manual = sum((xi - media_manual) ** 2 for xi in x) / (n_cfm - 1)
validar("Media exacta", media_manual, exactas["media"], 1e-6, "Σx/n vs np.mean")
validar("Varianza exacta", var_manual, exactas["varianza"], 1e-2, "Σ(x−x̄)²/(n−1) vs np.var(ddof=1)")
ordenados = np.sort(x)
mediana_manual = (ordenados[n_cfm // 2 - 1] + ordenados[n_cfm // 2]) / 2 if n_cfm % 2 == 0 else ordenados[n_cfm // 2]
validar("Mediana exacta", float(mediana_manual), exactas["mediana"], 0, "posición central ordenada vs np.median")
valores, conteos = np.unique(x, return_counts=True)
validar("Moda exacta", float(valores[np.argmax(conteos)]), exactas["moda"], 0, "conteo manual vs statistics.mode")

comparacion = pd.DataFrame({"agrupada": {k_: agrupadas[k_] for k_ in exactas}, "exacta": exactas})
comparacion["diferencia"] = comparacion.agrupada - comparacion.exacta
comparacion["diferencia_pct"] = 100 * comparacion.diferencia / comparacion.exacta
extra = {
    "ceros": int((x == 0).sum()), "valor_mas_frecuente_pct": float(conteos.max() / n_cfm),
    "distintos": int(len(valores)), "pct_en_clase_modal": float(f[i_mod] / n_cfm),
    "sobre_Q3": int((x > exactas["Q3"]).sum()), "hasta_Q1": int((x <= exactas["Q1"]).sum()),
    "pct_hasta_100k": float((x < 100_000).mean()),
}
print(extra)
comparacion.round(2)

{'ceros': 45, 'valor_mas_frecuente_pct': 0.6512428298279158, 'distintos': 135, 'pct_en_clase_modal': 0.7998087954110898, 'sobre_Q3': 904, 'hasta_Q1': 4326, 'pct_hasta_100k': 1.0}


,agrupada,exacta,diferencia,diferencia_pct
media,39661.76,39276.45,385.31,0.98
mediana,40642.71,40185.00,457.71,1.14
moda,41023.61,40185.00,838.61,2.09
Q1,38300.90,40185.00,-1884.10,-4.69
Q2,40642.71,40185.00,457.71,1.14
Q3,42984.52,40185.00,2799.52,6.97
varianza,40573848.16,41291877.73,-718029.57,-1.74
desviacion,6369.76,6425.88,-56.12,-0.87
cv,16.06,16.36,-0.30,-1.84


## 3. ANTIGUEDAD_MESES (variable discreta, 5.3.1)
Se describe con los valores individuales y se agrupa en tramos de permanencia útiles para fidelización
(no se trata como continua).

In [8]:
a = el["ANTIGUEDAD_MESES"].dropna().astype(int).to_numpy()
tramos = pd.cut(a, [-1, 3, 12, 36, 120, 10_000], labels=["0–3 meses", "4–12 meses", "13–36 meses", "37–120 meses", "Más de 120 meses"])
tabla_antig = pd.Series(tramos).value_counts().reindex(tramos.categories).rename("f").to_frame()
tabla_antig["h"] = tabla_antig.f / len(a)
tabla_antig["H"] = tabla_antig.h.cumsum()
antig = {"n": len(a), "faltantes": int(el.ANTIGUEDAD_MESES.isna().sum()), "media": float(a.mean()),
         "mediana": float(np.median(a)), "moda": int(statistics.mode(a)),
         "Q1": float(np.quantile(a, .25)), "Q3": float(np.quantile(a, .75)), "varianza": float(np.var(a, ddof=1)),
         "desviacion": float(np.std(a, ddof=1)), "min": int(a.min()), "max": int(a.max()),
         "pct_moda": float((a == statistics.mode(a)).mean())}
antig["cv"] = antig["desviacion"] / antig["media"] * 100
valores_a, conteos_a = np.unique(a, return_counts=True)
print(pd.Series(antig).round(2))
tabla_antig

n             5227.00
faltantes        3.00
media            9.12
mediana          1.00
moda             1.00
Q1               1.00
Q3               3.00
varianza       989.84
desviacion      31.46
min              0.00
max            296.00
pct_moda         0.46
cv             344.94
dtype: float64


,f,h,H
0–3 meses,4021,0.769275,0.769275
4–12 meses,771,0.147503,0.916778
13–36 meses,144,0.027549,0.944328
37–120 meses,141,0.026975,0.971303
Más de 120 meses,150,0.028697,1.000000


## 4. Proporciones de las banderas (5.3.3 a 5.3.6)
p̂ = f / n, con f = registros con el indicador activo y n = registros con información válida.

In [9]:
def proporcion(mascara):
    f_ = int(mascara.sum())
    return {"f": f_, "n": int(len(mascara)), "p": f_ / len(mascara)}


props = {b: proporcion(el[b] == True) for b in ["BAN_BANCA", "BAN_BILLETERA", "BAN_PAGOS_ONLINE", "BAN_SUPRESION_I"]}  # noqa: E712
props["BANCA_Y_PAGOS"] = proporcion((el.BAN_BANCA == True) & (el.BAN_PAGOS_ONLINE == True))  # noqa: E712
props["ALGUNA_FINANCIERA"] = proporcion(el[["BAN_BANCA", "BAN_BILLETERA", "BAN_PAGOS_ONLINE"]].any(axis=1))
contactables = el[el.BAN_SUPRESION_I == False]   # noqa: E712
props["CONTACTABLES"] = proporcion(el.BAN_SUPRESION_I == False)   # noqa: E712
props["BANCA_EN_CONTACTABLES"] = proporcion(contactables.BAN_BANCA == True)   # noqa: E712
props["BANCA_EN_RESTRINGIDOS"] = proporcion(el[el.BAN_SUPRESION_I == True].BAN_BANCA == True)   # noqa: E712
pd.DataFrame(props).T

,f,n,p
BAN_BANCA,341.0,5230.0,0.065201
BAN_BILLETERA,112.0,5230.0,0.021415
BAN_PAGOS_ONLINE,198.0,5230.0,0.037859
BAN_SUPRESION_I,4583.0,5230.0,0.876291
BANCA_Y_PAGOS,175.0,5230.0,0.033461
ALGUNA_FINANCIERA,368.0,5230.0,0.070363
CONTACTABLES,647.0,5230.0,0.123709
BANCA_EN_CONTACTABLES,194.0,647.0,0.299845
BANCA_EN_RESTRINGIDOS,147.0,4583.0,0.032075


### Por mes (alimenta el filtro del dashboard)
Las mismas frecuencias y proporciones separadas por mes de actualización. El dashboard recalcula con ellas las
tablas y medidas cuando se escoge un mes; con "Todos" coinciden con los resultados de arriba.

In [10]:
def props_de(df):
    c_ = df[df.BAN_SUPRESION_I == False]   # noqa: E712
    r_ = df[df.BAN_SUPRESION_I == True]    # noqa: E712
    out = {b: proporcion(df[b] == True) for b in ["BAN_BANCA", "BAN_BILLETERA", "BAN_PAGOS_ONLINE", "BAN_SUPRESION_I"]}  # noqa: E712
    out["BANCA_Y_PAGOS"] = proporcion((df.BAN_BANCA == True) & (df.BAN_PAGOS_ONLINE == True))  # noqa: E712
    out["CONTACTABLES"] = proporcion(df.BAN_SUPRESION_I == False)   # noqa: E712
    out["BANCA_EN_CONTACTABLES"] = proporcion(c_.BAN_BANCA == True) if len(c_) else {"f": 0, "n": 0, "p": None}  # noqa: E712
    out["BANCA_EN_RESTRINGIDOS"] = proporcion(r_.BAN_BANCA == True) if len(r_) else {"f": 0, "n": 0, "p": None}  # noqa: E712
    return out


por_mes = {}
for mes_, g_ in el.groupby("FECHA_ACTUALIZACION"):
    v_, c_ = np.unique(g_.VAL_CFM.dropna().to_numpy(), return_counts=True)
    por_mes[mes_] = {"nombre": config.MESES.get(mes_, mes_), "cfm_frecuencias": [[float(a_), int(b_)] for a_, b_ in zip(v_, c_)],
                     "proporciones": props_de(g_)}
assert sum(sum(c for _, c in d["cfm_frecuencias"]) for d in por_mes.values()) == n_cfm
pd.DataFrame({m_: {"n": d["proporciones"]["BAN_BANCA"]["n"], "banca": d["proporciones"]["BAN_BANCA"]["p"]} for m_, d in por_mes.items()}).T

,n,banca
2026-01,501.0,0.111776
2026-02,376.0,0.154255
2026-03,455.0,0.153846
2026-04,573.0,0.122164
2026-05,726.0,0.024793
2026-06,531.0,0.128060
2026-07,733.0,0.000000
2026-08,947.0,0.001056
2026-10,388.0,0.000000


In [11]:
config.guardar_json({
    "por_mes": por_mes,
    "n": n, "N_recibidos": N_recibidos, "clasificacion": clasificacion.to_dict("records"),
    "agrupados": {"n": n_cfm, "xmin": xmin, "xmax": xmax, "R": R, "k_sturges": k_sturges, "k": k, "A": A,
                  "tabla": tabla.to_dict("records"), "medidas": agrupadas},
    "exactas": exactas, "comparacion": comparacion.reset_index(names="medida").to_dict("records"), "extra_cfm": extra,
    "cfm_frecuencias": [[float(v), int(c)] for v, c in zip(valores, conteos)],   # distribución (sin identificadores)
    "antiguedad": antig, "antiguedad_tramos": tabla_antig.reset_index(names="tramo").to_dict("records"),
    "antiguedad_frecuencias": [[int(v), int(c)] for v, c in zip(valores_a, conteos_a)],
    "proporciones": props, "validacion": validacion,
}, "r01_descriptiva")
pd.DataFrame(validacion)

,indicador,manual,computacional,diferencia,tolerancia,cumple,observacion
0,Σ fᵢ = n,5.230000e+03,5.230000e+03,0.000000e+00,0.000000e+00,True,suma de frecuencias absolutas
1,Σ hᵢ = 1,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e-09,True,suma de frecuencias relativas
2,Media agrupada,3.966176e+04,3.966176e+04,0.000000e+00,1.000000e-06,True,lazo manual vs np.average con pesos
3,Varianza agrupada,4.057385e+07,4.057385e+07,7.450581e-09,1.000000e-03,True,lazo manual vs operación vectorizada
4,Media agrupada vs. marcas repetidas,3.966176e+04,3.966176e+04,0.000000e+00,1.000000e-06,True,np.repeat de las marcas
5,Media exacta,3.927645e+04,3.927645e+04,0.000000e+00,1.000000e-06,True,Σx/n vs np.mean
6,Varianza exacta,4.129188e+07,4.129188e+07,3.576279e-06,1.000000e-02,True,Σ(x−x̄)²/(n−1) vs np.var(ddof=1)
7,Mediana exacta,4.018500e+04,4.018500e+04,0.000000e+00,0.000000e+00,True,posición central ordenada vs np.median
8,Moda exacta,4.018500e+04,4.018500e+04,0.000000e+00,0.000000e+00,True,conteo manual vs statistics.mode
